# Joins

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-05-joins.ipynb)

Real databases split information across tables. `customers` knows each customer's name, `orders` knows each
order's amount, and the two are linked by `customer_id`. A **join** combines the tables by matching that key, so a
single query can answer questions that span both.

This notebook explains each join type, the rules that decide how many rows come back, and the mistakes that
most often produce wrong totals. It ends with practice questions and their answers.

## Setup

Install PySpark, start a Spark session, and create the example tables.

In [ ]:
!pip install -q pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customers AS
SELECT * FROM VALUES
  (1, 'Acme Corp',    'North', DATE '2022-01-05'),
  (2, 'Beta LLC',     'North', DATE '2022-02-10'),
  (3, 'Cyan Inc',     'South', DATE '2022-03-15'),
  (4, 'Delta Co',     'South', DATE '2022-04-20'),
  (5, 'Echo Ltd',     'West',  DATE '2022-05-25'),
  (6, 'Foxtrot GmbH', 'West',  DATE '2022-06-30'),
  (7, 'Gamma SA',     'East',  DATE '2023-01-10'),
  (8, 'Hotel BV',     'East',  DATE '2023-02-15')
AS customers(customer_id, name, region, signup_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW orders AS
SELECT * FROM VALUES
  (1,  1, 'completed', 100, DATE '2023-01-05'),
  (2,  1, 'completed', 200, DATE '2023-02-10'),
  (3,  1, 'completed', 150, DATE '2023-03-15'),
  (4,  2, 'completed', 300, DATE '2023-01-20'),
  (5,  2, 'cancelled', 250, DATE '2023-02-05'),
  (6,  2, 'completed', 400, DATE '2023-03-20'),
  (7,  3, 'completed', 120, DATE '2023-01-25'),
  (8,  3, 'completed', 180, DATE '2023-02-28'),
  (9,  4, 'pending',   220, DATE '2023-02-14'),
  (10, 4, 'completed', 320, DATE '2023-03-30'),
  (11, 5, 'completed', 500, DATE '2023-01-02'),
  (12, 5, 'completed', 450, DATE '2023-03-05'),
  (13, 6, 'completed', 275, DATE '2023-02-18'),
  (14, 6, 'cancelled', 325, DATE '2023-03-25')
AS orders(order_id, customer_id, status, amount, order_date)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW customer_tags AS
SELECT * FROM VALUES (1, 'vip'), (1, 'eu'), (2, 'vip') AS customer_tags(customer_id, tag)
""")

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW staff AS
SELECT * FROM VALUES
  (1, 'Alice', NULL),
  (2, 'Bob',   1),
  (3, 'Carol', 1),
  (4, 'Dave',  2),
  (5, 'Eve',   2)
AS staff(emp_id, name, manager_id)
""")

## 1. The join condition

Every join needs a **condition** that says which rows match. It is written after `ON`:

```sql
SELECT ...
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id;
```

Read it as: for each order, find the customer whose `customer_id` is the same. Table aliases (`o`, `c`) keep
the column references short and tell you which table each column comes from.

## 2. Inner join

`INNER JOIN` (or just `JOIN`) keeps only the rows that match in **both** tables. An order whose customer does not exist, or a customer with no orders, disappears from the result.

In [ ]:
spark.sql("""
SELECT o.order_id, c.name, o.amount
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
ORDER BY o.order_id
LIMIT 5
""").show()

Every one of the 14 orders matches a customer, so the join returns 14 rows. Customers 7 and 8 have no orders,
so they are not in the result.

## 3. Left join

`LEFT JOIN` keeps **every row of the left table**, even when nothing matches on the right. The right-hand
columns are filled with `NULL` for those rows. Use it when the left table is the list you want to report on.

The classic use is finding the unmatched rows: a `LEFT JOIN` followed by `WHERE right_key IS NULL`.

In [ ]:
spark.sql("""
SELECT c.customer_id, c.name, o.order_id
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE c.customer_id >= 7
ORDER BY c.customer_id
""").show()

## 4. Right join and full outer join

`RIGHT JOIN` is the mirror image of `LEFT JOIN`: it keeps every row of the right table. `FULL OUTER JOIN` keeps
every row of **both** tables. Unmatched rows from either side get `NULL`s for the other side's columns.

The small example below uses inline tables with one key on each side.

In [ ]:
spark.sql("""
WITH a AS (SELECT * FROM VALUES (1), (2) AS a(id)),
     b AS (SELECT * FROM VALUES (2), (3) AS b(id))
SELECT a.id AS left_id, b.id AS right_id
FROM a
FULL OUTER JOIN b ON a.id = b.id
ORDER BY COALESCE(a.id, b.id)
""").show()

## 5. Cross join

A `CROSS JOIN` pairs every row of the left table with every row of the right table. There is no condition. The
result has `left rows × right rows` rows, so use it deliberately, for example to build a grid of combinations.

In [ ]:
spark.sql("""
SELECT r.region, s.status
FROM (SELECT DISTINCT region FROM customers) r
CROSS JOIN (SELECT DISTINCT status FROM orders) s
ORDER BY r.region, s.status
""").show(12)

## 6. Semi join and anti join

Two joins return only columns from the left table:

- `LEFT SEMI JOIN` keeps each left row that has **at least one** match. It is the same as `WHERE EXISTS`.
- `LEFT ANTI JOIN` keeps each left row that has **no** match. It is the same as `WHERE NOT EXISTS`.

Neither can duplicate a row, which makes them safer than an inner join when you only need to test for a match.

In [ ]:
spark.sql("""
SELECT c.name
FROM customers c
LEFT SEMI JOIN orders o ON o.customer_id = c.customer_id
ORDER BY c.name
""").show()

spark.sql("""
SELECT c.name
FROM customers c
LEFT ANTI JOIN orders o ON o.customer_id = c.customer_id
ORDER BY c.name
""").show()

## 7. Conditions in ON versus WHERE

This is the mistake that trips up most people with outer joins.

- A condition on the right-hand table in `ON` decides which right rows can match. Left rows without a match are
  still kept.
- The same condition in `WHERE` runs after the join. It removes the `NULL` rows, which quietly turns a
  `LEFT JOIN` into an inner join.

The two queries below count completed orders per customer. Only the first keeps customers 7 and 8.

In [ ]:
spark.sql("""
SELECT c.customer_id, COUNT(o.order_id) AS completed_orders
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'completed'
GROUP BY c.customer_id
ORDER BY c.customer_id
""").show()

spark.sql("""
SELECT c.customer_id, COUNT(o.order_id) AS completed_orders
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.status = 'completed'
GROUP BY c.customer_id
ORDER BY c.customer_id
""").show()

## 8. Row multiplication: why totals can be wrong

A join returns one row for every match. When one customer matches several rows in another table, the customer's
orders are repeated, and a `SUM` counts them more than once.

`customer_tags` has two tags for customer 1. Joining orders to it repeats each of customer 1's orders twice. The
correct total of completed orders for customers 1 and 2 is 1,150, but the naive join reports 1,600.

The fix is to ask a membership question instead of joining: a semi join keeps each order once.

In [ ]:
spark.sql("""
SELECT SUM(o.amount) AS naive_total
FROM orders o
JOIN customer_tags t ON t.customer_id = o.customer_id
WHERE o.status = 'completed'
""").show()

spark.sql("""
SELECT SUM(o.amount) AS correct_total
FROM orders o
LEFT SEMI JOIN customer_tags t ON t.customer_id = o.customer_id
WHERE o.status = 'completed'
""").show()

## 9. Self join

A table can join to itself if each side gets its own alias. The `staff` table records each employee's manager, so
joining it to itself shows each employee next to their manager. `LEFT JOIN` keeps Alice, who has no manager.

In [ ]:
spark.sql("""
SELECT e.name AS employee, m.name AS manager
FROM staff e
LEFT JOIN staff m ON m.emp_id = e.manager_id
ORDER BY e.emp_id
""").show()

## Summary

- An **inner join** keeps matches only. A **left join** keeps every left row.
- **Full outer** keeps both sides; **cross** pairs everything; **semi** and **anti** test for a match.
- Put conditions on the right-hand table in `ON` when unmatched left rows must survive.
- Check the join's row count before you sum anything. One-to-many joins repeat the "one" side's amounts.

## Practice questions

1. List every order with the customer's name, sorted by `order_id`. Show the first three rows.
2. Which customers have no tags? Show their names, sorted by name.
3. For every customer, show the name and the number of tags. Include customers with no tags, sorted by `customer_id`.
4. Show each employee with their manager's name, sorted by `emp_id`. Alice has no manager.

## Answers

### Question 1

In [ ]:
q = """
SELECT o.order_id, c.name
FROM orders o
JOIN customers c ON c.customer_id = o.customer_id
ORDER BY o.order_id
"""
spark.sql(q).show(3)
assert [tuple(r) for r in spark.sql(q).limit(3).collect()] == [(1, "Acme Corp"), (2, "Acme Corp"), (3, "Acme Corp")]

### Question 2

In [ ]:
q = """
SELECT c.name
FROM customers c
LEFT ANTI JOIN customer_tags t ON t.customer_id = c.customer_id
ORDER BY c.name
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [("Cyan Inc",), ("Delta Co",), ("Echo Ltd",), ("Foxtrot GmbH",), ("Gamma SA",), ("Hotel BV",)]

### Question 3

In [ ]:
q = """
SELECT c.name, COUNT(t.tag) AS tags
FROM customers c
LEFT JOIN customer_tags t ON t.customer_id = c.customer_id
GROUP BY c.customer_id, c.name
ORDER BY c.customer_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    ("Acme Corp", 2), ("Beta LLC", 1), ("Cyan Inc", 0), ("Delta Co", 0),
    ("Echo Ltd", 0), ("Foxtrot GmbH", 0), ("Gamma SA", 0), ("Hotel BV", 0),
]

Counting `t.tag` rather than `*` gives 0 for customers without tags, because the `LEFT JOIN` fills their tag with `NULL`.

### Question 4

In [ ]:
q = """
SELECT e.name AS employee, m.name AS manager
FROM staff e
LEFT JOIN staff m ON m.emp_id = e.manager_id
ORDER BY e.emp_id
"""
spark.sql(q).show()
assert spark.sql(q).collect() == [
    ("Alice", None), ("Bob", "Alice"), ("Carol", "Alice"), ("Dave", "Bob"), ("Eve", "Bob"),
]